# Prophet Fundamentals

**Prophet** is Meta's open-source forecasting library.  Unlike ARIMA, which
models the autocorrelation structure of a time series, Prophet takes a
*decomposition* approach: it breaks a time series into interpretable
components and fits each one with a purpose-built model.

This notebook covers:
1. The decomposable model: $y(t) = g(t) + s(t) + h(t) + \varepsilon_t$
2. Trend component: piecewise linear and logistic growth
3. Seasonality component: Fourier series
4. Holiday/event effects
5. Data format requirements
6. Basic fitting workflow
7. Component plots
8. Application to airline passengers
9. Application to alcohol sales
10. Train/test evaluation

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Prophet — note: the package name is "prophet", NOT "fbprophet"
# pip install prophet
from prophet import Prophet

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. The Decomposable Model

Prophet models a time series as the sum of three components plus noise:

$$
y(t) = g(t) + s(t) + h(t) + \varepsilon_t
$$

| Component | What it captures | How it is modeled |
|-----------|-----------------|-------------------|
| $g(t)$ | **Trend** — long-term growth or decline | Piecewise linear or logistic growth |
| $s(t)$ | **Seasonality** — periodic patterns (yearly, weekly, etc.) | Truncated Fourier series |
| $h(t)$ | **Holidays / events** — irregular, predictable disruptions | Indicator variables with windows |
| $\varepsilon_t$ | **Error** — everything else | Assumed normally distributed |

This is similar in spirit to classical decomposition (Chapter 2), but
each component uses a flexible, parametric model rather than simple
moving averages.  The parameters are estimated by fitting a *curve*
to the data — not by analyzing autocorrelation as in ARIMA.

---
## 2. Trend Component: $g(t)$

Prophet offers two trend models.

### Piecewise Linear Trend (default)

The trend is a line whose *slope* can change at automatically detected
**changepoints**.  Between changepoints, the trend is linear:

$$
g(t) = \left(k + \mathbf{a}(t)^T \boldsymbol{\delta}\right) t + \left(m + \mathbf{a}(t)^T \boldsymbol{\gamma}\right)
$$

where:
- $k$ is the base growth rate
- $\boldsymbol{\delta}$ is a vector of rate adjustments at each changepoint
- $\mathbf{a}(t)$ is an indicator vector: $a_j(t) = 1$ if $t \geq s_j$ (changepoint $j$)
- $m$ is the offset, $\boldsymbol{\gamma}$ ensures continuity

In plain English: the trend is a connected sequence of line segments,
where the slope changes at each changepoint.

### Logistic Growth Trend

For series with a natural *carrying capacity* (e.g., market saturation):

$$
g(t) = \frac{C(t)}{1 + \exp\left(-(k + \mathbf{a}(t)^T \boldsymbol{\delta})(t - (m + \mathbf{a}(t)^T \boldsymbol{\gamma}))\right)}
$$

where $C(t)$ is the (potentially time-varying) capacity.  This requires
the user to specify a `cap` (and optionally a `floor`).

In [ ]:
# Visualize a piecewise linear trend to build intuition
t = np.linspace(0, 10, 500)

# Changepoints at t=3 and t=7
k_base = 1.0  # base slope
changepoints = [3.0, 7.0]
deltas = [-0.8, 0.5]  # slope adjustments

# Build the piecewise linear function
slope = np.full_like(t, k_base)
for cp, delta in zip(changepoints, deltas):
    slope = np.where(t >= cp, slope + delta, slope)

# Cumulative sum to get the trend (integral of slope)
trend = np.cumsum(slope) * (t[1] - t[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Slope over time
axes[0].plot(t, slope, color="tab:blue")
for cp in changepoints:
    axes[0].axvline(cp, color="red", linestyle="--", alpha=0.6, label="changepoint")
axes[0].set_title("Growth Rate (slope) Over Time")
axes[0].set_xlabel("t")
axes[0].set_ylabel("Slope $k(t)$")

# Resulting trend
axes[1].plot(t, trend, color="tab:green")
for cp in changepoints:
    axes[1].axvline(cp, color="red", linestyle="--", alpha=0.6)
axes[1].set_title("Resulting Piecewise Linear Trend $g(t)$")
axes[1].set_xlabel("t")
axes[1].set_ylabel("$g(t)$")

plt.tight_layout()
plt.show()

print("The slope changes at each changepoint, producing a connected")
print("sequence of line segments.  Prophet detects these automatically.")

---
## 3. Seasonality Component: $s(t)$ — Fourier Series

Prophet models seasonality using a **truncated Fourier series**.  Any
periodic function can be approximated by a sum of sines and cosines:

$$
s(t) = \sum_{n=1}^{N}\left(a_n \cos\frac{2\pi n t}{P} + b_n \sin\frac{2\pi n t}{P}\right)
$$

where:
- $P$ is the period (e.g., $P = 365.25$ days for yearly seasonality, $P = 7$ days for weekly)
- $N$ is the **Fourier order** — how many sine/cosine pairs to include
- $a_n, b_n$ are coefficients estimated from the data

**Higher $N$** means the seasonal pattern can have more wiggles (more
flexible), but risks overfitting.  **Lower $N$** produces smoother
seasonal patterns.

### Prophet defaults

| Seasonality | Period $P$ | Default Fourier order $N$ |
|-------------|-----------|--------------------------|
| Yearly      | 365.25    | 10                       |
| Weekly      | 7         | 3                        |
| Daily       | 1         | 4                        |

In [ ]:
# Visualize Fourier series with different orders
t = np.linspace(0, 365.25 * 2, 1000)  # two years of "days"
P = 365.25

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, N in zip(axes.flat, [1, 3, 6, 10]):
    # Build a Fourier series with some fixed coefficients
    np.random.seed(42)
    a = np.random.randn(N) / np.arange(1, N + 1)  # decay with n
    b = np.random.randn(N) / np.arange(1, N + 1)

    s = np.zeros_like(t)
    for n in range(1, N + 1):
        s += a[n - 1] * np.cos(2 * np.pi * n * t / P)
        s += b[n - 1] * np.sin(2 * np.pi * n * t / P)

    ax.plot(t / 365.25, s, color="tab:purple")
    ax.set_title(f"Fourier order $N = {N}$")
    ax.set_xlabel("Years")
    ax.set_ylabel("$s(t)$")
    ax.axhline(0, color="grey", linestyle="--", alpha=0.4)

plt.suptitle("Fourier Series Approximations of Yearly Seasonality", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("N=1: simple sinusoid.  N=10: complex seasonal shape.")
print("Prophet defaults to N=10 for yearly seasonality.")

---
## 4. Holiday / Event Effects: $h(t)$

Holidays and special events create disruptions that are not periodic.
Prophet handles them with indicator variables:

$$
h(t) = \sum_{i} \kappa_i \cdot \mathbf{1}_{t \in D_i}
$$

where $D_i$ is the set of dates in the window around holiday $i$, and
$\kappa_i$ is the estimated effect size.

Each holiday can have:
- `lower_window`: how many days *before* the holiday to include (e.g., -1 for the day before)
- `upper_window`: how many days *after* the holiday to include (e.g., 1 for the day after)

Prophet also supports adding all holidays for a given country with
`model.add_country_holidays(country_name='US')`.

---
## 5. Data Format Requirements

Prophet is strict about its input format.  It requires a DataFrame with
exactly two columns:

| Column | Description | Example |
|--------|-------------|---------|
| `ds` | Datestamp — must be a date or datetime | `2020-01-15` |
| `y` | Value to forecast — numeric | `42.7` |

No index-based dates, no other column names.  This is the single most
common source of errors when starting with Prophet.

In [ ]:
# Load airline passengers data and prepare for Prophet
airline = pd.read_csv(DATA_DIR / "airline_passengers.csv")
print("Raw columns:", airline.columns.tolist())
airline.head()

In [ ]:
# Rename columns to Prophet's required format: 'ds' and 'y'
airline_prophet = airline.rename(columns={
    "Month": "ds",
    "Thousands of Passengers": "y",
})
airline_prophet["ds"] = pd.to_datetime(airline_prophet["ds"])

print(f"Shape: {airline_prophet.shape}")
print(f"Date range: {airline_prophet['ds'].min()} to {airline_prophet['ds'].max()}")
airline_prophet.head()

In [ ]:
# Quick look at the data
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(airline_prophet["ds"], airline_prophet["y"], marker="o", markersize=3)
ax.set_title("Airline Passengers (1949-1960)")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

print("Clear upward trend + increasing seasonal amplitude (multiplicative).")
print("We will start with default additive Prophet and see how it does.")

---
## 6. Basic Fitting Workflow

The Prophet workflow follows a simple pattern:
1. Create a `Prophet()` object
2. Call `model.fit(df)` with the training data
3. Create a future DataFrame with `model.make_future_dataframe()`
4. Generate predictions with `model.predict(future)`
5. Visualize with `model.plot()` and `model.plot_components()`

In [ ]:
# Train/test split — hold out last 24 months
n_test = 24
train_airline = airline_prophet.iloc[:-n_test].copy()
test_airline = airline_prophet.iloc[-n_test:].copy()

print(f"Train: {len(train_airline)} months ({train_airline['ds'].iloc[0].date()} to {train_airline['ds'].iloc[-1].date()})")
print(f"Test:  {len(test_airline)} months ({test_airline['ds'].iloc[0].date()} to {test_airline['ds'].iloc[-1].date()})")

In [ ]:
# Step 1 & 2: Create and fit the model
model_airline = Prophet()
model_airline.fit(train_airline)

In [ ]:
# Step 3: Create a future DataFrame
# periods=24 to cover the test period; freq='ME' for month-end
future_airline = model_airline.make_future_dataframe(periods=n_test, freq="ME")

print(f"Future DataFrame shape: {future_airline.shape}")
print(f"Includes training dates + {n_test} forecast dates")
future_airline.tail()

In [ ]:
# Step 4: Generate predictions
forecast_airline = model_airline.predict(future_airline)

# The forecast DataFrame has many columns — the key ones:
print("Key forecast columns:")
print(forecast_airline[["ds", "yhat", "yhat_lower", "yhat_upper", "trend", "yearly"]].tail())

In [ ]:
# Step 5a: Plot the forecast
fig = model_airline.plot(forecast_airline)
plt.title("Prophet Forecast — Airline Passengers (Default Settings)")
plt.ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

print("Black dots = observed data.  Blue line = yhat (prediction).")
print("Light blue band = uncertainty interval.")

---
## 7. Component Plots

One of Prophet's strengths is interpretability.  The `plot_components()`
method shows each component of the decomposition separately:
- **Trend**: the long-term growth pattern
- **Yearly seasonality**: the repeating annual pattern

This is the $g(t) + s(t)$ decomposition in action.

In [ ]:
# Step 5b: Component plots
fig = model_airline.plot_components(forecast_airline)
plt.tight_layout()
plt.show()

print("Trend: steady upward growth in air travel.")
print("Yearly: peaks in summer (Jul-Aug), troughs in winter (Nov-Feb).")

---
## 8. Forecast vs Actuals

Let us overlay the Prophet forecast against the held-out test data to
see how well the default model performs.

In [ ]:
# Extract forecast for the test period
forecast_test = forecast_airline[forecast_airline["ds"].isin(test_airline["ds"])].copy()

fig, ax = plt.subplots(figsize=(14, 5))

# Training data
ax.plot(train_airline["ds"], train_airline["y"], color="tab:blue", label="Train")

# Test actuals
ax.plot(test_airline["ds"], test_airline["y"], color="tab:green", label="Test (actual)", marker="o", markersize=4)

# Prophet forecast
ax.plot(forecast_test["ds"], forecast_test["yhat"], color="tab:red", label="Prophet forecast", linestyle="--")
ax.fill_between(
    forecast_test["ds"],
    forecast_test["yhat_lower"],
    forecast_test["yhat_upper"],
    alpha=0.2, color="tab:red", label="Uncertainty interval",
)

ax.set_title("Airline Passengers — Prophet Forecast vs Actuals")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compute forecast accuracy metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = test_airline["y"].values
predicted = forecast_test["yhat"].values

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
mape = np.mean(np.abs((actual - predicted) / actual)) * 100

print("Default Prophet — Airline Passengers (24-month holdout)")
print(f"  MAE:  {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAPE: {mape:.2f}%")
print()
print("The default additive model misses the increasing seasonal amplitude.")
print("We will fix this with multiplicative seasonality in the tuning notebook.")

---
## 9. Understanding the Forecast DataFrame

Prophet's `predict()` returns a rich DataFrame with all components.
Let us examine its structure.

In [ ]:
# All columns in the forecast DataFrame
print(f"Number of columns: {len(forecast_airline.columns)}")
print()
print("Key columns:")
key_cols = {
    "ds": "datestamp",
    "yhat": "point forecast (trend + seasonality + holidays)",
    "yhat_lower": "lower bound of uncertainty interval",
    "yhat_upper": "upper bound of uncertainty interval",
    "trend": "trend component g(t)",
    "trend_lower": "trend uncertainty lower",
    "trend_upper": "trend uncertainty upper",
    "yearly": "yearly seasonality s(t)",
    "yearly_lower": "yearly seasonality uncertainty lower",
    "yearly_upper": "yearly seasonality uncertainty upper",
}
for col, desc in key_cols.items():
    if col in forecast_airline.columns:
        print(f"  {col:20s} — {desc}")

In [ ]:
# Verify: yhat ≈ trend + yearly (for additive model with no holidays)
check = forecast_airline[["ds", "yhat", "trend", "yearly"]].tail(10).copy()
check["trend_plus_yearly"] = check["trend"] + check["yearly"]
check["difference"] = check["yhat"] - check["trend_plus_yearly"]
print(check.to_string(index=False))
print()
print("The difference is negligible — yhat = trend + yearly (additive decomposition).")

---
## 10. Visualizing Components Manually

We can extract and plot each component ourselves for a more customized view.

In [ ]:
# Custom component visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Original data + forecast
axes[0].plot(airline_prophet["ds"], airline_prophet["y"],
             color="black", alpha=0.5, label="Observed")
axes[0].plot(forecast_airline["ds"], forecast_airline["yhat"],
             color="tab:blue", label="Forecast ($\\hat{y}$)")
axes[0].set_title("Data + Forecast")
axes[0].set_ylabel("Passengers (thousands)")
axes[0].legend()

# Trend component
axes[1].plot(forecast_airline["ds"], forecast_airline["trend"],
             color="tab:green", linewidth=2)
axes[1].fill_between(
    forecast_airline["ds"],
    forecast_airline["trend_lower"],
    forecast_airline["trend_upper"],
    alpha=0.2, color="tab:green",
)
axes[1].set_title("Trend Component $g(t)$")
axes[1].set_ylabel("Trend")

# Yearly seasonality
axes[2].plot(forecast_airline["ds"], forecast_airline["yearly"],
             color="tab:purple")
axes[2].axhline(0, color="grey", linestyle="--", alpha=0.5)
axes[2].set_title("Yearly Seasonality $s(t)$")
axes[2].set_ylabel("Seasonal effect")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()

---
## 11. Application to Alcohol Sales

Let us apply Prophet to a second dataset — monthly alcohol sales — which
has a longer history and different characteristics.

In [ ]:
# Load and prepare alcohol sales data
alcohol = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv")
print("Raw columns:", alcohol.columns.tolist())
alcohol.head()

In [ ]:
# Prepare for Prophet
alcohol_prophet = alcohol.rename(columns={
    "DATE": "ds",
    "S4248SM144NCEN": "y",
})
alcohol_prophet["ds"] = pd.to_datetime(alcohol_prophet["ds"])

print(f"Shape: {alcohol_prophet.shape}")
print(f"Date range: {alcohol_prophet['ds'].min().date()} to {alcohol_prophet['ds'].max().date()}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(alcohol_prophet["ds"], alcohol_prophet["y"])
ax.set_title("Monthly Alcohol Sales")
ax.set_xlabel("Date")
ax.set_ylabel("Millions of Dollars")
plt.tight_layout()
plt.show()

print("Upward trend with strong yearly seasonality (December spikes).")
print("Seasonal amplitude increases with the level — multiplicative pattern.")

In [ ]:
# Train/test split — hold out last 36 months
n_test_alc = 36
train_alcohol = alcohol_prophet.iloc[:-n_test_alc].copy()
test_alcohol = alcohol_prophet.iloc[-n_test_alc:].copy()

print(f"Train: {len(train_alcohol)} months")
print(f"Test:  {len(test_alcohol)} months")

In [ ]:
# Fit Prophet with default settings
model_alcohol = Prophet()
model_alcohol.fit(train_alcohol)

future_alcohol = model_alcohol.make_future_dataframe(periods=n_test_alc, freq="MS")
forecast_alcohol = model_alcohol.predict(future_alcohol)

In [ ]:
# Plot forecast
fig = model_alcohol.plot(forecast_alcohol)
plt.title("Prophet Forecast — Alcohol Sales (Default Settings)")
plt.ylabel("Millions of Dollars")
plt.tight_layout()
plt.show()

In [ ]:
# Component plots for alcohol sales
fig = model_alcohol.plot_components(forecast_alcohol)
plt.tight_layout()
plt.show()

print("Trend: steady upward growth in alcohol sales.")
print("Yearly: strong December spike (holiday season buying).")

In [ ]:
# Forecast vs actuals for alcohol sales
forecast_test_alc = forecast_alcohol.set_index("ds").loc[test_alcohol["ds"].values].reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_alcohol["ds"], train_alcohol["y"], color="tab:blue", label="Train")
ax.plot(test_alcohol["ds"], test_alcohol["y"], color="tab:green",
        label="Test (actual)", marker="o", markersize=4)
ax.plot(forecast_test_alc["ds"], forecast_test_alc["yhat"], color="tab:red",
        label="Prophet forecast", linestyle="--")
ax.fill_between(
    forecast_test_alc["ds"],
    forecast_test_alc["yhat_lower"],
    forecast_test_alc["yhat_upper"],
    alpha=0.2, color="tab:red",
)
ax.set_title("Alcohol Sales — Prophet Default vs Actuals")
ax.set_xlabel("Date")
ax.set_ylabel("Millions of Dollars")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy metrics for alcohol sales
actual_alc = test_alcohol["y"].values
pred_alc = forecast_test_alc["yhat"].values

mae_alc = mean_absolute_error(actual_alc, pred_alc)
rmse_alc = np.sqrt(mean_squared_error(actual_alc, pred_alc))
mape_alc = np.mean(np.abs((actual_alc - pred_alc) / actual_alc)) * 100

print("Default Prophet — Alcohol Sales (36-month holdout)")
print(f"  MAE:  {mae_alc:.2f}")
print(f"  RMSE: {rmse_alc:.2f}")
print(f"  MAPE: {mape_alc:.2f}%")

---
## 12. Inspecting Prophet's Internal Parameters

We can access the fitted model's internal parameters to understand
what Prophet learned.

In [ ]:
# Changepoints detected by Prophet
print(f"Number of potential changepoints: {len(model_airline.changepoints)}")
print(f"\nChangepoint dates (first 10):")
for i, cp in enumerate(model_airline.changepoints[:10]):
    print(f"  {i+1}. {cp.date()}")

print(f"\nChangepoint range: Prophet places changepoints in the first "
      f"{model_airline.changepoint_range * 100:.0f}% of the training data.")

In [ ]:
# Examine the rate changes (deltas) at each changepoint
params = model_airline.params
deltas = params["delta"].flatten()

print(f"Base growth rate (k): {params['k'][0]:.6f}")
print(f"Number of rate changes: {len(deltas)}")
print(f"\nLargest rate changes (absolute):")
top_idx = np.argsort(np.abs(deltas))[::-1][:5]
for idx in top_idx:
    if idx < len(model_airline.changepoints):
        print(f"  {model_airline.changepoints.iloc[idx].date()}: "
              f"delta = {deltas[idx]:+.6f}")

---
## 13. Residual Analysis

Even though Prophet is not a statistical model in the ARIMA sense, we
should still check whether the residuals look like white noise.

In [ ]:
# Compute residuals for training period
from statsmodels.graphics.tsaplots import plot_acf

forecast_train = forecast_airline[forecast_airline["ds"].isin(train_airline["ds"])].copy()
residuals = train_airline["y"].values - forecast_train["yhat"].values

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residual plot
axes[0].plot(train_airline["ds"].values, residuals, linewidth=0.8)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Residuals Over Time")
axes[0].set_xlabel("Date")

# Histogram
axes[1].hist(residuals, bins=20, edgecolor="black", alpha=0.7)
axes[1].set_title("Residual Distribution")
axes[1].axvline(0, color="red", linestyle="--")

# ACF of residuals
plot_acf(residuals, lags=24, ax=axes[2], title="ACF of Residuals")

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f}")
print(f"Std residual:  {residuals.std():.2f}")
print()
print("If significant autocorrelation remains in the residuals,")
print("Prophet is not fully capturing the data's structure.")

---
## 14. Prophet vs Simple Approaches

Let us compare Prophet's default forecast to a naive seasonal baseline
to understand whether it adds value beyond simple methods.

In [ ]:
# Seasonal naive: repeat the last year's values
seasonal_naive = train_airline["y"].values[-12:]  # last 12 months of training
# Tile to cover 24 months of test
seasonal_naive_forecast = np.tile(seasonal_naive, n_test // 12 + 1)[:n_test]

mae_naive = mean_absolute_error(actual, seasonal_naive_forecast)
rmse_naive = np.sqrt(mean_squared_error(actual, seasonal_naive_forecast))
mape_naive = np.mean(np.abs((actual - seasonal_naive_forecast) / actual)) * 100

print("Comparison — Airline Passengers (24-month holdout)")
print(f"{'Method':<25s} {'MAE':>8s} {'RMSE':>8s} {'MAPE':>8s}")
print("-" * 52)
print(f"{'Seasonal Naive':<25s} {mae_naive:8.2f} {rmse_naive:8.2f} {mape_naive:7.2f}%")
print(f"{'Prophet (default)':<25s} {mae:8.2f} {rmse:8.2f} {mape:7.2f}%")
print()
print("Prophet captures the upward trend that seasonal naive misses.")

---
## Summary

- Prophet models time series as $y(t) = g(t) + s(t) + h(t) + \varepsilon_t$
  - $g(t)$: piecewise linear or logistic growth trend
  - $s(t)$: Fourier series for seasonality
  - $h(t)$: holiday indicator effects
  - $\varepsilon_t$: normally distributed error

- **Data format**: must be a DataFrame with columns `ds` (date) and `y` (value)

- **Workflow**: `Prophet()` → `.fit(df)` → `.make_future_dataframe()` → `.predict()` → `.plot()` / `.plot_components()`

- The **Fourier series** for seasonality uses $N$ sine/cosine pairs:
  $s(t) = \sum_{n=1}^{N}(a_n \cos \frac{2\pi nt}{P} + b_n \sin \frac{2\pi nt}{P})$

- Prophet **automatically detects changepoints** in the trend

- The forecast DataFrame contains all components (`trend`, `yearly`, etc.)
  and uncertainty intervals (`yhat_lower`, `yhat_upper`)

- Default additive Prophet works well out of the box, but the airline
  passengers data shows that **multiplicative seasonality** may be needed
  when seasonal amplitude grows with the level

In [ ]:
print("Next notebook: Prophet tuning — changepoints, seasonality modes,")
print("holidays, and logistic growth.")